In [3]:
import sqlite3
import pandas as pd

df = pd.read_csv("Churn.csv")

conn = sqlite3.connect("churn.db")
df.to_sql("customers", conn, if_exists="replace", index=False)

print("Loaded successfully")
print(pd.read_sql("SELECT COUNT(*) as row_count FROM customers", conn))

Loaded successfully
   row_count
0      10000


In [4]:
query1 = """
SELECT 
    Geography,
    COUNT(*) AS total_customers,
    SUM(Exited) AS churned_customers,
    ROUND(100.0 * SUM(Exited) / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY Geography
ORDER BY churn_rate_pct DESC;
"""
print(pd.read_sql(query1, conn))

  Geography  total_customers  churned_customers  churn_rate_pct
0   Germany             2509                814           32.44
1     Spain             2477                413           16.67
2    France             5014                811           16.17


Churn rate by Geography

In [5]:
query2 = """
SELECT 
    Tenure,
    COUNT(*) AS total_customers,
    SUM(Exited) AS churned_customers,
    ROUND(100.0 * SUM(Exited) / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY Tenure
ORDER BY Tenure;
"""
print(pd.read_sql(query2, conn))

    Tenure  total_customers  churned_customers  churn_rate_pct
0        0              413                 95           23.00
1        1             1035                232           22.42
2        2             1048                201           19.18
3        3             1009                213           21.11
4        4              989                203           20.53
5        5             1012                209           20.65
6        6              967                196           20.27
7        7             1028                177           17.22
8        8             1025                197           19.22
9        9              984                214           21.75
10      10              490                101           20.61


Churn rate by Tenure

In [6]:
query3 = """
SELECT 
    Geography,
    Gender,
    COUNT(*) AS total_customers,
    ROUND(100.0 * SUM(Exited) / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY Geography, Gender
ORDER BY churn_rate_pct DESC;
"""
print(pd.read_sql(query3, conn))

  Geography  Gender  total_customers  churn_rate_pct
0   Germany  Female             1193           37.55
1   Germany    Male             1316           27.81
2     Spain  Female             1089           21.21
3    France  Female             2261           20.34
4     Spain    Male             1388           13.11
5    France    Male             2753           12.75


Churn rate by Geography + Gender

In [7]:
query4 = """
SELECT 
    CustomerId,
    Geography,
    Balance,
    Exited,
    RANK() OVER (PARTITION BY Geography ORDER BY Balance DESC) AS balance_rank
FROM customers
ORDER BY Geography, balance_rank
LIMIT 15;
"""
print(pd.read_sql(query4, conn))

    CustomerId Geography    Balance  Exited  balance_rank
0     15715622    France  238387.56       1             1
1     15769818    France  212778.20       0             2
2     15780212    France  212692.97       0             3
3     15690589    France  212314.03       0             4
4     15671256    France  211774.31       1             5
5     15736420    France  210433.08       1             6
6     15709920    France  208165.53       1             7
7     15627971    France  206663.75       0             8
8     15784180    France  206329.65       1             9
9     15664498    France  205962.00       0            10
10    15673020    France  204510.94       1            11
11    15620756    France  202904.64       1            12
12    15620570    France  202443.47       0            13
13    15689514    France  201696.07       1            14
14    15793688    France  201009.64       1            15


Rank customers by Balance within each Geography (window function)

In [8]:
query5 = """
SELECT 
    CASE 
        WHEN NumOfProducts >= 3 THEN 'High Risk (3-4 products)'
        WHEN NumOfProducts = 1 THEN 'Medium Risk (1 product)'
        ELSE 'Low Risk (2 products)'
    END AS risk_segment,
    COUNT(*) AS total_customers,
    ROUND(100.0 * SUM(Exited) / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY risk_segment
ORDER BY churn_rate_pct DESC;
"""
print(pd.read_sql(query5, conn))

               risk_segment  total_customers  churn_rate_pct
0  High Risk (3-4 products)              326           85.89
1   Medium Risk (1 product)             5084           27.71
2     Low Risk (2 products)             4590            7.60


Risk segment via CASE WHEN (encodes your NumOfProducts finding)